# Association Rule Mining for E-Commerce Recommendations

| Key              | Value                                                                                                                                                                                                                                                                                                        |
|:-----------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Course Codes** | BBT 4206 and BFS 4102                                                                                                                                                                                                                                                                                        |
| **Course Names** | BBT 4206: Business Intelligence II (Week 1-3 of 13) and<br/>BFS 4102: Advanced Business Data Analytics (Week 4-6 of 13)                                                                                                                                                                                      |
| **Semester**     | September to November 2026                                                                                                                                                                                                                                                                                   |
| **Lecturer**     | Allan Omondi                                                                                                                                                                                                                                                                                                 |
| **Contact**      | aomondi@strathmore.edu                                                                                                                                                                                                                                                                                       |
| **Note**         | The lecture contains both theory and practice.<br/>This notebook forms part of the practice.<br/>It is intended for educational purposes only.<br/>Recommended citation: [BibTex](https://raw.githubusercontent.com/course-files/ClusteringandAssociationRuleMining/refs/heads/main/RecommendedCitation.bib) |

**Business context**: *Soko Safi*, an online retailer, wants to add a "Frequently Bought
Together" widget to its product pages and power a basic recommendation engine at checkout.
Association rule mining (the technique behind the classic "market basket analysis") is the
natural starting point: it finds which products are actually purchased together often enough,
and strongly enough, to justify recommending one when a customer has the other in their cart.

**What you are required to do**: starting from raw order data in the format a real e-commerce
database would actually export it (one row per item purchased identified using SKUs, not a
pre-bundled list) you are required to construct transaction baskets, mine frequent
itemsets and rules, validate that those rules hold up on data the mining
process never saw, and turn the surviving rules into a working recommendation function.

**What is an SKU?** An **SKU (Stock Keeping Unit)** is the unique code a retailer assigns to
each distinct product it sells — for example, `E01` for a specific phone case. Real systems
mine rules on SKUs, not product names, for a simple production reason: a product's *name* can
be spelled inconsistently across systems, translated, or renamed during a promotion, but its
*SKU* is a stable, canonical identifier. Product names are reserved for the final step —
translating a rule back into something a human (or a webpage) can read.

**Remote Environments:**

Do your best to setup your local environment as guided during the lab, however, if you have challenges setting it up, then you can use the following remote environments temporarily for the lab:<br/>

[![Colab](https://img.shields.io/badge/Open-Colab-orange?logo=googlecolab)](
https://colab.research.google.com/github/course-files/ClusteringandAssociationRuleMining/blob/main/4_association_rule_mining.ipynb) (preferred option)

[![Codespaces](https://img.shields.io/badge/Open-Codespaces-blue?logo=github)](
https://github.com/codespaces/new/course-files/ClusteringandAssociationRuleMining) (alternative)


## 1. Install Dependencies

In [ ]:
# Uncomment if any of these are not yet installed
# %pip install pandas numpy mlxtend matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
RANDOM_STATE = 42

# For suppressing warnings displayed in the notebook
import warnings
warnings.filterwarnings('ignore')

print("Environment ready.")

## 2. Load the Data

Real order data arrives as **line items**, not ready-made transactions: one row per product
purchased, tagged with the order it belongs to. This is deliberately what you get here.

In [ ]:
# `parse_dates=["order_date"]` tells pandas to parse the `order_date` column as
# datetime64 values instead of leaving it as plain strings (object dtype). This:
# - Enables date/time operations — you can perform tasks like orders["order_date"].dt.month,
#   .dt.day_name(), filtering with orders[orders["order_date"] > "2024-01-01"], or
#   resampling/grouping by time period (e.g., orders.resample("M", on="order_date")).
# - Correct sorting — sorting by date works chronologically rather than as a string order.
# - Correct comparisons/arithmetic — you can subtract dates to get durations, compare dates, etc.
#
# Without parse_dates, order_date would be read as plain text (strings), and
# you would have to manually convert it later using
# pd.to_datetime(orders["order_date"]) to get the same functionality.
orders = pd.read_csv("./data/ecommerce_orders.csv", parse_dates=["order_date"])

print(f"Shape: {orders.shape[0]} order lines, {orders.shape[1]} columns")
print(f"Unique orders: {orders['order_id'].nunique()}")
print(f"Unique SKUs: {orders['sku'].nunique()}")
print(f"Date range: {orders['order_date'].min().date()} to {orders['order_date'].max().date()}")
orders.head(8)

**Notes on Modelling Decisions:**

`order_id` and `customer_id` are identifiers, not products, so they will not
appear inside any basket (shopping cart). `quantity` and `unit_price_kes` exist
in real order data but are not needed for *which items co-occur*, which is what
association rule mining asks; therefore, we set them aside here as part of
feature selection.

## 3. Build Transactions from Raw Order Lines

One of the first things to do in this workflow is to **group line items by order**
to build each transaction's basket of SKUs.

In [ ]:
# - `orders.groupby("order_id")["sku"]`: groups rows by each order and selects the sku column.

# - `.apply(list)`: converts each group's SKUs into a list.
# So each order_id becomes something like:
# order 1001 → ["E01", "E02", "E04"]
# order 1002 → ["P01", "P02"]

# `.tolist()`: takes the pandas Series of lists and converts it into one regular Python list.
# So the final transactions look like this:
# [
#   ['E02', 'H06', 'E04']
#   ['O01', 'O02', 'O03']
#   ['H03', 'E02', 'F05']
#   ['E02', 'P05', 'P03', 'F07']
# ]
transactions = orders.groupby("order_id")["sku"].apply(list).tolist()

print(f"Number of transactions (orders): {len(transactions)}")
print("\nFirst 5 transactions:")
for t in transactions[:5]:
    print(f"  {t}")

basket_sizes = orders.groupby("order_id").size()
print(f"\nBasket size: mean={basket_sizes.mean():.2f}, min={basket_sizes.min()}, max={basket_sizes.max()}")

We also build a **SKU → product name/category lookup table**. This is a standard
production pattern: mine and compute on the stable identifier (SKU), and only
translate to a human-readable name at the point of reporting or display; never
the other way around.

In [ ]:
sku_lookup = orders.drop_duplicates("sku").set_index("sku")[["product_name", "category", "unit_price_kes"]]
sku_lookup.head()

## 4. Simplified Example: How Apriori Works

**For learning purposes:**
Before running this on thousands of orders, we work through the mechanics on a
handful of transactions you can manually confirm. **This step matters**: a
parameter like `min_support` only makes sense relative to how many transactions you
have. A `min_support` copied from a 5,000-transaction analysis onto a 6-transaction
example would accept literally everything — the parameter must be chosen relative to the data in front of you, every time.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

demo_transactions = transactions[:6]
print("Demo transactions:")
for t in demo_transactions:
    print(f"  {t}")

demo_encoder = TransactionEncoder()
demo_onehot = demo_encoder.fit(demo_transactions).transform(demo_transactions)
demo_data = pd.DataFrame(demo_onehot, columns=demo_encoder.columns_)
demo_data

In [ ]:
# With only 6 transactions, we can decide that an itemset must appear in at
# least 2 of them (support >= 0.333) to imply a significant association rule.
demo_min_support = 2 / len(demo_transactions)
print(f"Using min_support = {demo_min_support:.3f} (>= 2 of {len(demo_transactions)} transactions)")

demo_freq = apriori(demo_data, min_support=demo_min_support, use_colnames=True)
demo_freq.sort_values("support", ascending=False)

**The lesson to carry forward**: always ask "how many transactions does this threshold
actually require?" before trusting a `min_support` value; on this scale, on the full
dataset's scale, or on any new dataset you may use later in your career.

## 5. Generate Frequent Itemsets on the Full Dataset

With 5,000 transactions, `min_support=0.01` requires an itemset to appear in at least 50
orders. This is a manually chosen threshold, not a universal constant.

**A production note on scale**: this catalog has only 35 SKUs, where Apriori's exhaustive
itemset search is instant. A real e-commerce catalog can have thousands of SKUs, where
Apriori's runtime grows sharply. `fpgrowth()` (from the same `mlxtend` library) finds
identical results using a different search strategy that scales better. In this dataset,
the difference is negligible, but you will notice the difference on a larger catalog.

In [ ]:
import time

encoder = TransactionEncoder()
onehot = encoder.fit(transactions).transform(transactions)
transaction_data = pd.DataFrame(onehot, columns=encoder.columns_)

MIN_SUPPORT = 0.01
print(f"min_support={MIN_SUPPORT} requires an itemset in at least {MIN_SUPPORT * len(transactions):.0f} of {len(transactions)} orders.\n")

t0 = time.time()
frequent_itemsets = apriori(transaction_data, min_support=MIN_SUPPORT, use_colnames=True)
t_apriori = time.time() - t0

from mlxtend.frequent_patterns import fpgrowth
t0 = time.time()
frequent_itemsets_fpgrowth = fpgrowth(transaction_data, min_support=MIN_SUPPORT, use_colnames=True)
t_fpgrowth = time.time() - t0

print(f"Apriori:   {len(frequent_itemsets)} frequent itemsets in {t_apriori:.4f}s")
print(f"FP-Growth: {len(frequent_itemsets_fpgrowth)} frequent itemsets in {t_fpgrowth:.4f}s")
print("\n(Both find the same itemsets at this scale -- the difference only matters as the catalog grows.)")

frequent_itemsets.sort_values("support", ascending=False).head(10)

## 6. Generate Association Rules

Recall the three key metrics:
- **Support**: a measure used to identify the frequency with which a
  particular item set (a specific set of products or services) appears in a
  transaction data set.
- **Confidence**: quantifies how often the items in the antecedent (the “if”
  part of the rule) and the items in the consequent (the “then” part of the
  rule) appear together in the dataset compared to the frequency of the
  antecedent alone
- **Lift**: a measure of how much more often the items in the antecedent (the
  “if” part of the rule) and the items in the consequent (the “then” part of
  the rule) appear together in transactions compared to what would be expected
  if they were statistically independent. A lift value greater than 1 suggests
  that the items in X and Y appear together more frequently than if they were
  independent.

In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print(f"Total rules generated: {len(rules)}")
rules.sort_values("confidence", ascending=False).head(10)[
    ["antecedents", "consequents", "support", "confidence", "lift"]
]

## 7. Choosing "Strong Rule" Thresholds

A `confidence >= 0.8` threshold might be sensible on one dataset and eliminate *every single
rule* on another; thresholds are properties of the data, not universal constants.
We, therefore, look at the actual distribution before picking a cutoff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(rules["confidence"], bins=20, ax=axes[0], color="#4F6EA4")
axes[0].set_title("Distribution of Confidence")
sns.histplot(rules["lift"], bins=20, ax=axes[1], color="#4F6EA4")
axes[1].set_title("Distribution of Lift")
plt.tight_layout()
plt.show()

print(rules[["confidence", "lift"]].describe().round(3))

**Notes on modelling decisions:**

Thresholds chosen with reference to the distribution above (roughly the 60th percentile of
confidence, and a lift comfortably above 1 to require a genuinely strong association).

In [ ]:
CONFIDENCE_THRESHOLD = 0.5
LIFT_THRESHOLD = 3.0

strong_rules = rules[
    (rules["confidence"] >= CONFIDENCE_THRESHOLD) & (rules["lift"] >= LIFT_THRESHOLD)
].sort_values(["lift", "confidence"], ascending=[False, False])

print(f"Strong rules (confidence >= {CONFIDENCE_THRESHOLD}, lift >= {LIFT_THRESHOLD}): {len(strong_rules)}")
strong_rules.head(10)[["antecedents", "consequents", "support", "confidence", "lift"]]

## 8. Remove Redundant Rules

Two kinds of redundancy are worth removing before acting on these rules:
1. A rule is redundant if a **simpler** rule (smaller antecedent) already implies it with
   equal or better lift/confidence.
2. A rule and its reverse (`{A} → {B}` and `{B} → {A}`) are often the same discovery stated
   twice, for a "frequently bought together" use case where direction does not matter.

In [ ]:
def remove_subset_redundant_rules(rules_df):
    """Remove a rule if a simpler (subset-antecedent) rule with the same consequent
    already captures it at equal or higher lift/confidence."""
    rules_df = rules_df.sort_values(["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)
    kept_rules = []
    for _, row in rules_df.iterrows():
        is_redundant = False
        for kept in kept_rules:
            if row["consequents"] == kept["consequents"] and row["antecedents"].issuperset(kept["antecedents"]) and row["antecedents"] != kept["antecedents"]:
                is_redundant = True
                break
        if not is_redundant:
            kept_rules.append(row)
    return pd.DataFrame(kept_rules)


def remove_bidirectional_redundancy(rules_df):
    """Keep only one direction of a {A}->{B} / {B}->{A} pair (the higher-lift one, since
    sorting was already applied upstream)."""
    seen_pairs = set()
    kept_rows = []
    for _, row in rules_df.iterrows():
        pair_key = frozenset([frozenset(row["antecedents"]), frozenset(row["consequents"])])
        if pair_key not in seen_pairs:
            seen_pairs.add(pair_key)
            kept_rows.append(row)
    return pd.DataFrame(kept_rows)


nonredundant_rules = remove_subset_redundant_rules(strong_rules)
cleaned_rules = remove_bidirectional_redundancy(nonredundant_rules)
cleaned_rules = cleaned_rules.sort_values(["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)

print(f"Strong rules: {len(strong_rules)} -> after removing subset redundancy: {len(nonredundant_rules)} "
      f"-> after removing bidirectional redundancy: {len(cleaned_rules)}")
cleaned_rules[["antecedents", "consequents", "support", "confidence", "lift"]]

## 9. Translate Rules into Human-Readable Product Names

The mining is done on SKUs but the report is provided to humans.
This translation step is where SKUs are mapped to product names.

In [ ]:
def skus_to_names(sku_frozenset):
    return ", ".join(sorted(sku_lookup.loc[list(sku_frozenset), "product_name"]))

readable_rules = cleaned_rules.copy()
readable_rules["antecedent_products"] = readable_rules["antecedents"].apply(skus_to_names)
readable_rules["consequent_products"] = readable_rules["consequents"].apply(skus_to_names)

readable_rules[["antecedent_products", "consequent_products", "support", "confidence", "lift"]].head(12)

## 10. Validate Rules on a Held-Out Time Period

Every rule so far was mined and immediately trusted on the *same* data it came from. This
is similar to evaluating a predictive model only on its training set. A rule
with strong in-sample confidence might just be a coincidence of this particular data snapshot.
Here, rules are mined on an earlier period and checked against a **later period the mining
process never saw**, mirroring the train/test discipline from the regression and
classification labs we did, but adapted to a setting with no single held-out metric to check.

In [ ]:
CUTOFF_DATE = pd.Timestamp("2025-05-15")
train_orders = orders[orders["order_date"] < CUTOFF_DATE]
validation_orders = orders[orders["order_date"] >= CUTOFF_DATE]

print(f"Training period orders:   {train_orders['order_id'].nunique()} (before {CUTOFF_DATE.date()})")
print(f"Validation period orders: {validation_orders['order_id'].nunique()} (on/after {CUTOFF_DATE.date()})")

train_transactions = train_orders.groupby("order_id")["sku"].apply(list).tolist()
validation_transactions = validation_orders.groupby("order_id")["sku"].apply(list).tolist()

train_encoder = TransactionEncoder()
train_onehot = pd.DataFrame(train_encoder.fit(train_transactions).transform(train_transactions),
                             columns=train_encoder.columns_)
train_freq = apriori(train_onehot, min_support=MIN_SUPPORT, use_colnames=True)
train_rules = association_rules(train_freq, metric="lift", min_threshold=1.0)
train_strong = train_rules[
    (train_rules["confidence"] >= CONFIDENCE_THRESHOLD) & (train_rules["lift"] >= LIFT_THRESHOLD)
]
print(f"\nStrong rules mined on the training period alone: {len(train_strong)}")

In [ ]:
validation_encoder = TransactionEncoder()
validation_onehot = pd.DataFrame(
    validation_encoder.fit(validation_transactions).transform(validation_transactions),
    columns=validation_encoder.columns_
)

def confidence_on(onehot_df, antecedent, consequent):
    """Recompute confidence for a specific antecedent/consequent pair on a given onehot dataset."""
    antecedent, consequent = list(antecedent), list(consequent)
    if not all(a in onehot_df.columns for a in antecedent) or not all(c in onehot_df.columns for c in consequent):
        return np.nan, 0
    antecedent_mask = onehot_df[antecedent].all(axis=1)
    both_mask = antecedent_mask & onehot_df[consequent].all(axis=1)
    n_antecedent = antecedent_mask.sum()
    return (both_mask.sum() / n_antecedent if n_antecedent > 0 else np.nan), n_antecedent

validation_results = []
for _, row in train_strong.iterrows():
    val_confidence, n_ante = confidence_on(validation_onehot, row["antecedents"], row["consequents"])
    validation_results.append({
        "antecedents": row["antecedents"], "consequents": row["consequents"],
        "train_confidence": row["confidence"], "validation_confidence": val_confidence,
        "validation_support_count": n_ante,
    })

validation_df = pd.DataFrame(validation_results)
validation_df["confidence_drop"] = validation_df["train_confidence"] - validation_df["validation_confidence"]
validation_df = validation_df.sort_values("confidence_drop", ascending=False)

print("Rules with the largest drop in confidence from training to validation period (check these first):")
validation_df.head(8).round(3)

A small, evenly scattered drop in confidence (a few percentage points either direction)
is normal sampling noise from splitting the data. A rule whose confidence collapses on the
validation period (even though it looked strong in training) should be treated with
suspicion before it goes anywhere near a live recommendation engine; it may reflect a
coincidence in the training window rather than a genuine, stable customer behavior.

In [ ]:
large_drop_threshold = 0.15
unstable_rules = validation_df[validation_df["confidence_drop"].abs() > large_drop_threshold]
print(f"Rules with a confidence shift greater than {large_drop_threshold} between periods: {len(unstable_rules)}")
if len(unstable_rules) == 0:
    print("None -- the rules mined on the training period generalize consistently to the validation period.")

## 11. Visualize the Rules

In [ ]:
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=rules, x="support", y="confidence", size="lift", hue="lift",
                 palette="viridis", sizes=(20, 200))
plt.title("Support vs. Confidence (bubble size/color = lift)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=rules, x="lift", y="confidence", size="support", hue="support",
                 palette="viridis", sizes=(20, 200))
plt.title("Lift vs. Confidence (bubble size/color = support)")
plt.tight_layout()
plt.show()

## 12. From Rules to Recommendations

### 12.1 The naive approach, and why it doesn't scale

The simplest implementation hardcodes each rule as an `if` statement. This is fine for a
two-line demo and wrong for production: it only returns the *first* matching rule, ignores
every other applicable rule, and requires a code change every time the rules are refreshed.

In [ ]:
def recommend_naive(cart):
    if {"E01", "E02"}.issubset(cart):
        return "E04"
    elif {"O01", "O02"}.issubset(cart):
        return "O03"
    else:
        return None

# This only ever returns ONE recommendation and silently ignores every other rule that
# might also apply to the same cart.
print(recommend_naive({"E01", "E02"}))

### 12.2 A production-style recommender: a precomputed lookup table

A real recommendation service needs to respond in milliseconds, for arbitrary carts, without
looping through a rules table on every request. The standard fix is to **precompute a lookup
dictionary once** (antecedent → ranked list of consequents), so that serving a recommendation
becomes a single dictionary lookup rather than a scan.

In [ ]:
def build_recommendation_lookup(rules_df):
    """Precompute antecedent (frozenset of SKUs) -> list of (consequent_sku, confidence, lift),
    sorted by confidence. This is the structure an actual recommendation service would hold
    in memory, refreshed each time the rules are re-mined."""
    lookup = {}
    for _, row in rules_df.iterrows():
        key = frozenset(row["antecedents"])
        for consequent_sku in row["consequents"]:
            lookup.setdefault(key, []).append((consequent_sku, row["confidence"], row["lift"]))
    for key in lookup:
        lookup[key] = sorted(lookup[key], key=lambda x: x[1], reverse=True)
    return lookup


recommendation_lookup = build_recommendation_lookup(cleaned_rules)
print(f"Precomputed lookup covers {len(recommendation_lookup)} distinct antecedent combinations.")

In [ ]:
def recommend(cart, lookup, sku_lookup_table, top_n=3):
    """Return up to top_n recommended SKUs (with product names) for a cart, drawn from every
    matching rule -- not just the first one -- ranked by confidence, excluding items already
    in the cart."""
    cart = set(cart)
    candidates = {}
    for antecedent, consequent_list in lookup.items():
        if antecedent.issubset(cart):
            for sku, confidence, lift in consequent_list:
                if sku not in cart:
                    if sku not in candidates or confidence > candidates[sku][0]:
                        candidates[sku] = (confidence, lift)

    ranked = sorted(candidates.items(), key=lambda x: x[1][0], reverse=True)[:top_n]
    return [
        {"sku": sku, "product_name": sku_lookup_table.loc[sku, "product_name"],
         "confidence": round(conf, 3), "lift": round(lift, 2)}
        for sku, (conf, lift) in ranked
    ]


example_cart = {"H06", "H07"}
# example_cart = {"P04", "P01"}
# example_cart = {"E01", "E02"}
recommendations = recommend(example_cart, recommendation_lookup, sku_lookup, top_n=3)
print(f"Cart: {[sku_lookup.loc[s, 'product_name'] for s in example_cart]}")
print("Recommendations:")
for rec in recommendations:
    print(f"  {rec['product_name']} (SKU {rec['sku']}) -- confidence={rec['confidence']}, lift={rec['lift']}")

## 13. Limitations of a Pure Association-Rule Recommender

Association rule mining is only one input to a recommender system, not a complete one.

- **Cold start**: a brand-new SKU with no purchase history cannot appear as a consequent in
  any rule, no matter how good a fit it might be — it has to earn its way into the rules over
  time, which is a real problem for a retailer launching new products.
- **No personalization**: every customer with the same cart gets the same recommendation.
  There is no notion of *this specific customer's* preferences here, unlike collaborative
  filtering or content-based approaches.
- **Popularity bias**: high-support items dominate frequent itemsets by construction, which
  can crowd out genuinely interesting but rarer associations unless thresholds are chosen
  carefully (Step 7).
- **Static until re-mined**: rules reflect the transaction window they were mined from
  (Step 10 made this concrete). A production system needs a defined refresh cadence — daily
  or weekly re-mining is typical for a fast-moving catalog.

In practice, association rules are usually combined with collaborative filtering and/or
content-based methods, not used alone.

## 14. Business Analysis

The cleaned rules point to clear, actionable bundles: `Phone Case` + `Screen Protector` →
`Fast Phone Charger`; `Laptop Bag` + `Wireless Mouse` → `Laptop Stand`; `Digital Camera` +
`Memory Card` → `Camera Bag`, among others. These are strong enough (lift well above 3)
to justify concrete action, not just observation:

- **Product page widgets**: present the top 1 to 3 recommendations from Step 12 directly on each
  product's page as "Frequently Bought Together."
- **Bundle pricing**: the Office and Photography clusters in particular show very high mutual
  confidence. This implies strong candidates for an actual discounted bundle SKU, not just a
  recommendation. Recall the clustering lab → should the bundle be offered to a specific cluster?
- **Checkout upsell**: use the same `recommend()` function against the current cart at
  checkout, not just on individual product pages.
- **Refresh cadence**: re-run Steps 5-10 on a rolling basis (e.g., monthly) and re-check
  Step 10's validation before replacing the live rule set. A rule that stops validating is a
  signal to investigate.

## 15. Persistence

In this case, what gets saved for production is not a single "model" object the way a regression or
classification pipeline has one — it is the **cleaned rules table** and the **precomputed
recommendation lookup** built from it, since that lookup is what actually serves
recommendations at request time.

In [ ]:
import joblib
import os
from datetime import datetime

# Ensure the model folder exists
model_dir = './model'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Build filename with a timestamp
# This timestamp pattern is useful beyond this one script.
# It prevents accidentally overwriting a previous run's saved rules.
# It also helps you to keep track of rule-set versions as the catalog changes.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
rules_filename = f"{timestamp}_apriori_recommendation_rules.joblib".lower()
rules_path = os.path.normpath(os.path.join(model_dir, rules_filename))

joblib.dump({
    "cleaned_rules": cleaned_rules,
    "recommendation_lookup": recommendation_lookup,
    "sku_lookup": sku_lookup,
}, rules_path)

# Human-readable CSV export, saved alongside the .joblib with the same timestamp
cleaned_rules_export = cleaned_rules.copy()
cleaned_rules_export["antecedents"] = cleaned_rules_export["antecedents"].apply(lambda s: ", ".join(sorted(s)))
cleaned_rules_export["consequents"] = cleaned_rules_export["consequents"].apply(lambda s: ", ".join(sorted(s)))
csv_filename = f"{timestamp}_top_association_rules.csv".lower()
csv_path = os.path.normpath(os.path.join(model_dir, csv_filename))
cleaned_rules_export.to_csv(csv_path, index=False)

print(f"✅ Saved rules + lookup + SKU catalog to: {rules_path}")
print(f"✅ Saved human-readable rules export to: {csv_path}")

In [ ]:
# Demonstrate loading and using the persisted lookup for a brand-new cart -- exactly what a
# recommendation microservice would do on each incoming request.
loaded = joblib.load("./model/apriori_recommendation_rules.joblib")
loaded_lookup = loaded["recommendation_lookup"]
loaded_sku_lookup = loaded["sku_lookup"]

# new_cart = {"H06", "H07"}
new_cart = {"P04", "P01"}
# new_cart = {"E01", "E02"}

new_recommendations = recommend(new_cart, loaded_lookup, loaded_sku_lookup, top_n=3)

print(f"New cart: {[loaded_sku_lookup.loc[s, 'product_name'] for s in new_cart]}")
print("Recommendations from the persisted, reloaded rule set:")
for rec in new_recommendations:
    print(f"  {rec['product_name']} (SKU {rec['sku']}) -- confidence={rec['confidence']}, lift={rec['lift']}")

## Summary

This notebook took e-commerce order data from its raw, production-realistic form (line items
identified by SKU) through to a working recommendation function: building transaction
baskets, mining frequent itemsets and rules with thresholds justified from the data's own
distribution rather than copied from elsewhere, removing redundant rules, translating SKUs to
product names for reporting, validating rule stability on a held-out time period, and
persisting a precomputed lookup structure suited to actually serving recommendations.

**The habit worth preserving in your career**: every threshold in this notebook (`min_support`, confidence,
lift, the validation cutoff date) was chosen by looking at *this* dataset's own distribution,
not copied from an existing notebook. The single biggest risk in association rule mining is
treating a threshold as a universal constant rather than a property of the data in front of
you.